In [1]:
!pip install numpy pandas scikit-learn

In [2]:
TRAIN_PATH = "/kaggle/input/datasets/vubinhminh/absa-dataset/preprocessed_train.csv"
DEV_PATH   = "/kaggle/input/datasets/vubinhminh/absa-dataset/preprocessed_dev.csv"
TEST_PATH  = "/kaggle/input/datasets/vubinhminh/absa-dataset/preprocessed_test.csv"

OUTPUT_DIR = "/kaggle/working/outputs"
MODEL_DIR = "/kaggle/working/outputs/models"
METRIC_DIR = "/kaggle/working/outputs/metrics"
PREDICTION_DIR = "/kaggle/working/outputs/predictions"

VECTORIZER_PATH = "/kaggle/working/outputs/models/tfidf_vectorizer.pkl"
CLASSIFIERS_PATH = "/kaggle/working/outputs/models/aspect_classifiers.pkl"
LABEL_ENCODERS_PATH = "/kaggle/working/outputs/models/label_encoders.pkl"

DEV_METRICS_PATH = "/kaggle/working/outputs/metrics/dev_metrics.json"
TEST_METRICS_PATH = "/kaggle/working/outputs/metrics/test_metrics.json"
TEST_PREDICTIONS_PATH = "/kaggle/working/outputs/predictions/test_predictions.csv"

In [3]:
TEXT_COLUMN = "clean_comment"

ASPECTS = [
    "GENERAL",
    "SCREEN",
    "CAMERA",
    "FEATURES",
    "BATTERY",
    "PERFORMANCE",
    "STORAGE",
    "DESIGN",
    "PRICE",
    "SER&ACC",
]

LABELS = ["none", "positive", "neutral", "negative"]

LABEL_MAP = {
    "0": "none",
    "1": "positive",
    "2": "negative",
    "3": "neutral",

    "none": "none",
    "nan": "none",
    "null": "none",
    "": "none",

    "positive": "positive",
    "pos": "positive",
    "p": "positive",

    "negative": "negative",
    "neg": "negative",
    "n": "negative",

    "neutral": "neutral",
    "neu": "neutral",
    "o": "neutral",
}

TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.95,
    "sublinear_tf": True,
    "strip_accents": None,
}

LOGREG_PARAMS = {
    "max_iter": 2000,
    "class_weight": "balanced",
    "solver": "liblinear",
    "random_state": 42,
}


In [4]:
import os
import json
import pickle
from typing import Dict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder

In [5]:
def get_preprocess_function():
    try:
        import preprocessing

        for fn_name in ["preprocess_text", "normalize_text", "clean_text"]:
            if hasattr(preprocessing, fn_name):
                print(f"Using preprocessing.{fn_name}()")
                return getattr(preprocessing, fn_name)

    except Exception as e:
        print(f"Could not import preprocessing.py. Using raw text. Reason: {e}")

    print("No preprocessing function found. Using raw text.")
    return lambda x: str(x)


In [6]:
def ensure_dirs():
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(METRIC_DIR, exist_ok=True)
    os.makedirs(PREDICTION_DIR, exist_ok=True)


def normalize_label(value) -> str:
    """Normalize dataset labels to none / positive / neutral / negative."""
    if pd.isna(value):
        return "none"

    value = str(value).strip().lower()

    if value in LABEL_MAP:
        return LABEL_MAP[value]

    raise ValueError(f"Unknown label value: {value}. Please update LABEL_MAP.")


def validate_columns(df: pd.DataFrame, split_name: str):
    required_columns = [TEXT_COLUMN] + ASPECTS
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            f"{split_name} is missing columns: {missing}\n"
            f"Available columns: {list(df.columns)}\n"
            f"Please update TEXT_COLUMN and ASPECTS."
        )


def load_dataset(path: str, split_name: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{split_name} file not found: {path}\n"
            f"Please paste the correct Kaggle path into the path variables."
        )

    df = pd.read_csv(path)
    validate_columns(df, split_name)

    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

    for aspect in ASPECTS:
        df[aspect] = df[aspect].apply(normalize_label)

    return df


def prepare_texts(df: pd.DataFrame) -> np.ndarray:
    preprocess_fn = get_preprocess_function()
    texts = df[TEXT_COLUMN].apply(preprocess_fn).fillna("").astype(str)
    return texts.values


def save_pickle(obj, path: str):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Model artifact not found: {path}\n"
            f"Please run the training cells first."
        )

    with open(path, "rb") as f:
        return pickle.load(f)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


In [7]:
ensure_dirs()

train_df = load_dataset(TRAIN_PATH, "train")
dev_df = load_dataset(DEV_PATH, "dev")
test_df = load_dataset(TEST_PATH, "test")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (7786, 16)
Dev shape: (1112, 16)
Test shape: (2224, 16)


,index,comment,n_star,date_time,label,clean_comment,GENERAL,SCREEN,CAMERA,FEATURES,BATTERY,PERFORMANCE,STORAGE,DESIGN,PRICE,SER&ACC
0,0,Mới mua máy này Tại thegioididong thốt nốt cảm...,5,2 tuần trước,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,mới mua máy này tại thegioididong thốt nốt cảm...,positive,none,positive,positive,positive,none,none,none,positive,positive
1,1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,5,14/09/2019,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,pin kém còn lại miễn chê mua 8 3 2019 tình trạ...,positive,none,none,none,negative,none,none,none,none,none
2,2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,3,17/08/2020,{FEATURES#Negative};,sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,none,none,none,negative,none,none,none,none,none,none
3,3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",3,29/02/2020,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,mọi người cập nhật phần mềm lại nó sẽ bớt tốn ...,neutral,none,none,negative,neutral,none,none,none,none,none
4,4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",5,4/6/2020,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,mới mua xài được 1 tháng thấy pin rất trâu xài...,none,none,none,none,positive,positive,none,none,none,negative


In [8]:
for aspect in ASPECTS:
    print("\n" + "=" * 80)
    print(f"Aspect: {aspect}")
    print(train_df[aspect].value_counts(dropna=False))


Aspect: GENERAL
GENERAL
positive    3627
none        2920
negative     949
neutral      290
Name: count, dtype: int64

Aspect: SCREEN
SCREEN
none        6837
positive     514
negative     379
neutral       56
Name: count, dtype: int64

Aspect: CAMERA
CAMERA
none        5640
positive    1231
negative     627
neutral      288
Name: count, dtype: int64

Aspect: FEATURES
FEATURES
none        5144
negative    1659
positive     785
neutral      198
Name: count, dtype: int64

Aspect: BATTERY
BATTERY
none        4182
positive    2027
negative    1228
neutral      349
Name: count, dtype: int64

Aspect: PERFORMANCE
PERFORMANCE
none        3646
positive    2253
negative    1496
neutral      391
Name: count, dtype: int64

Aspect: STORAGE
STORAGE
none        7695
positive      59
negative      21
neutral       11
Name: count, dtype: int64

Aspect: DESIGN
DESIGN
none        6408
positive     999
negative     302
neutral       77
Name: count, dtype: int64

Aspect: PRICE
PRICE
none        5725
neutra

In [9]:
print("Preprocessing texts...")
train_texts = prepare_texts(train_df)
dev_texts = prepare_texts(dev_df)
test_texts = prepare_texts(test_df)

print("Training TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
vectorizer.fit(train_texts)

x_train = vectorizer.transform(train_texts)
x_dev = vectorizer.transform(dev_texts)
x_test = vectorizer.transform(test_texts)

print("x_train shape:", x_train.shape)
print("x_dev shape:", x_dev.shape)
print("x_test shape:", x_test.shape)

Preprocessing texts...
Could not import preprocessing.py. Using raw text. Reason: No module named 'preprocessing'
No preprocessing function found. Using raw text.
Could not import preprocessing.py. Using raw text. Reason: No module named 'preprocessing'
No preprocessing function found. Using raw text.
Could not import preprocessing.py. Using raw text. Reason: No module named 'preprocessing'
No preprocessing function found. Using raw text.
Training TF-IDF vectorizer...
x_train shape: (7786, 28978)
x_dev shape: (1112, 28978)
x_test shape: (2224, 28978)


In [10]:
classifiers = {}
label_encoders = {}

for aspect in ASPECTS:
    print(f"\nTraining classifier for aspect: {aspect}")

    y_text = train_df[aspect].values

    encoder = LabelEncoder()
    encoder.fit(LABELS)
    y_train = encoder.transform(y_text)

    clf = LogisticRegression(**LOGREG_PARAMS)
    clf.fit(x_train, y_train)

    classifiers[aspect] = clf
    label_encoders[aspect] = encoder

print("\nTraining completed.")


Training classifier for aspect: GENERAL

Training classifier for aspect: SCREEN

Training classifier for aspect: CAMERA

Training classifier for aspect: FEATURES

Training classifier for aspect: BATTERY

Training classifier for aspect: PERFORMANCE

Training classifier for aspect: STORAGE

Training classifier for aspect: DESIGN

Training classifier for aspect: PRICE

Training classifier for aspect: SER&ACC

Training completed.


In [11]:
def predict_all_aspects(x, classifiers: Dict, label_encoders: Dict) -> pd.DataFrame:
    predictions = {}

    for aspect in ASPECTS:
        clf = classifiers[aspect]
        encoder = label_encoders[aspect]

        pred_ids = clf.predict(x)
        pred_labels = encoder.inverse_transform(pred_ids)

        predictions[aspect] = pred_labels

    return pd.DataFrame(predictions)


def predict_probabilities(x, classifiers: Dict) -> pd.DataFrame:
    confidence = {}

    for aspect in ASPECTS:
        clf = classifiers[aspect]

        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba(x)
            confidence[f"{aspect}_confidence"] = probs.max(axis=1)
        else:
            confidence[f"{aspect}_confidence"] = np.nan

    return pd.DataFrame(confidence)


def evaluate(gold_df: pd.DataFrame, pred_df: pd.DataFrame, split_name: str, print_report: bool = False) -> Dict:
    metrics = {
        "split": split_name,
        "per_aspect": {},
    }

    macro_f1_scores = []
    weighted_f1_scores = []
    accuracy_scores = []

    for aspect in ASPECTS:
        y_true = gold_df[aspect].values
        y_pred = pred_df[aspect].values

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        )
        weighted_f1 = f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0,
        )
        acc = accuracy_score(y_true, y_pred)

        macro_f1_scores.append(macro_f1)
        weighted_f1_scores.append(weighted_f1)
        accuracy_scores.append(acc)

        if print_report:
            print("\n" + "=" * 80)
            print(f"Aspect: {aspect}")
            print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

        metrics["per_aspect"][aspect] = {
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "classification_report": classification_report(
                y_true,
                y_pred,
                labels=LABELS,
                zero_division=0,
                output_dict=True,
            ),
        }

    gold_matrix = gold_df[ASPECTS].values
    pred_matrix = pred_df[ASPECTS].values
    exact_match = np.mean(np.all(gold_matrix == pred_matrix, axis=1))

    metrics["overall"] = {
        "average_accuracy": float(np.mean(accuracy_scores)),
        "average_macro_f1": float(np.mean(macro_f1_scores)),
        "average_weighted_f1": float(np.mean(weighted_f1_scores)),
        "exact_match_ratio": float(exact_match),
    }

    return metrics


def print_overall_metrics(metrics: Dict):
    print(f"\n{metrics['split'].upper()} overall metrics:")
    for key, value in metrics["overall"].items():
        print(f"{key}: {value:.4f}")


In [12]:
dev_predictions = predict_all_aspects(x_dev, classifiers, label_encoders)
dev_metrics = evaluate(dev_df, dev_predictions, "dev", print_report=False)

print_overall_metrics(dev_metrics)
save_json(dev_metrics, DEV_METRICS_PATH)

print(f"\nSaved dev metrics to: {DEV_METRICS_PATH}")



DEV overall metrics:
average_accuracy: 0.8794
average_macro_f1: 0.5842
average_weighted_f1: 0.8693
exact_match_ratio: 0.3112

Saved dev metrics to: /kaggle/working/outputs/metrics/dev_metrics.json


In [13]:
save_pickle(vectorizer, VECTORIZER_PATH)
save_pickle(classifiers, CLASSIFIERS_PATH)
save_pickle(label_encoders, LABEL_ENCODERS_PATH)

print(f"Saved vectorizer to: {VECTORIZER_PATH}")
print(f"Saved classifiers to: {CLASSIFIERS_PATH}")
print(f"Saved label encoders to: {LABEL_ENCODERS_PATH}")

Saved vectorizer to: /kaggle/working/outputs/models/tfidf_vectorizer.pkl
Saved classifiers to: /kaggle/working/outputs/models/aspect_classifiers.pkl
Saved label encoders to: /kaggle/working/outputs/models/label_encoders.pkl


In [14]:
vectorizer = load_pickle(VECTORIZER_PATH)
classifiers = load_pickle(CLASSIFIERS_PATH)
label_encoders = load_pickle(LABEL_ENCODERS_PATH)

x_test = vectorizer.transform(test_texts)

print("Loaded saved artifacts successfully.")

Loaded saved artifacts successfully.


In [15]:
test_predictions = predict_all_aspects(x_test, classifiers, label_encoders)
confidence_df = predict_probabilities(x_test, classifiers)

test_metrics = evaluate(test_df, test_predictions, "test", print_report=True)

print_overall_metrics(test_metrics)
save_json(test_metrics, TEST_METRICS_PATH)

print(f"\nSaved test metrics to: {TEST_METRICS_PATH}")


Aspect: GENERAL
              precision    recall  f1-score   support

        none       0.72      0.72      0.72       843
    positive       0.83      0.84      0.83      1004
     neutral       0.44      0.35      0.39        83
    negative       0.66      0.69      0.68       294

    accuracy                           0.75      2224
   macro avg       0.66      0.65      0.65      2224
weighted avg       0.75      0.75      0.75      2224


Aspect: SCREEN
              precision    recall  f1-score   support

        none       0.95      0.98      0.97      1955
    positive       0.75      0.68      0.71       136
     neutral       0.00      0.00      0.00        17
    negative       0.68      0.58      0.63       116

    accuracy                           0.93      2224
   macro avg       0.60      0.56      0.58      2224
weighted avg       0.92      0.93      0.93      2224


Aspect: CAMERA
              precision    recall  f1-score   support

        none       0.93   

In [16]:
output_df = pd.DataFrame()
output_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].values

for aspect in ASPECTS:
    output_df[f"{aspect}_gold"] = test_df[aspect].values
    output_df[f"{aspect}_pred"] = test_predictions[aspect].values

    conf_col = f"{aspect}_confidence"
    if conf_col in confidence_df.columns:
        output_df[conf_col] = confidence_df[conf_col].values

output_df.to_csv(TEST_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

print(f"Saved test predictions to: {TEST_PREDICTIONS_PATH}")
output_df.head()

Saved test predictions to: /kaggle/working/outputs/predictions/test_predictions.csv


,clean_comment,GENERAL_gold,GENERAL_pred,GENERAL_confidence,SCREEN_gold,SCREEN_pred,SCREEN_confidence,CAMERA_gold,CAMERA_pred,CAMERA_confidence,...,STORAGE_confidence,DESIGN_gold,DESIGN_pred,DESIGN_confidence,PRICE_gold,PRICE_pred,PRICE_confidence,SER&ACC_gold,SER&ACC_pred,SER&ACC_confidence
0,điện thoải ổn facelock cực nhanh vân tay ôk mà...,positive,positive,0.525628,positive,positive,0.538709,none,none,0.579885,...,0.941203,none,none,0.673760,none,none,0.699448,none,none,0.849945
1,mình mới mua vivo91c tải ứng dụng games nhanh ...,none,none,0.597613,none,none,0.471257,none,none,0.817690,...,0.958651,none,none,0.814137,none,none,0.905429,positive,positive,0.519765
2,xấu đẹp gì không biết nhưng rất ưng tgdđ phục ...,none,positive,0.700210,none,none,0.827589,none,none,0.758920,...,0.951728,neutral,none,0.549731,none,none,0.858872,positive,positive,0.786899
3,màn hình hơi lác khi chơi game game nặng thì m...,none,none,0.769150,none,none,0.541828,none,none,0.802801,...,0.963039,negative,none,0.510848,none,none,0.881719,none,none,0.731602
4,nói chung máy đẹp với màn amoled ổn trong tầm ...,none,positive,0.599483,positive,none,0.556001,none,none,0.601677,...,0.906740,positive,positive,0.499024,none,neutral,0.715158,none,none,0.753442


In [17]:
rows = []

for aspect in ASPECTS:
    rows.append({
        "aspect": aspect,
        "accuracy": test_metrics["per_aspect"][aspect]["accuracy"],
        "macro_f1": test_metrics["per_aspect"][aspect]["macro_f1"],
        "weighted_f1": test_metrics["per_aspect"][aspect]["weighted_f1"],
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values("macro_f1", ascending=False)

summary_df

,aspect,accuracy,macro_f1,weighted_f1
8,PRICE,0.882194,0.710266,0.874807
4,BATTERY,0.861511,0.701921,0.854567
2,CAMERA,0.897032,0.695957,0.891026
5,PERFORMANCE,0.788219,0.654241,0.779205
0,GENERAL,0.753147,0.654003,0.751847
3,FEATURES,0.855665,0.598643,0.843667
1,SCREEN,0.930755,0.576246,0.925113
9,SER&ACC,0.880845,0.556862,0.866244
7,DESIGN,0.901079,0.509384,0.882993
6,STORAGE,0.990108,0.420239,0.987167
